In [15]:
import pandas as pd
import numpy as np
import joblib
import os
import warnings
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, f1_score
from imblearn.under_sampling import RandomUnderSampler
from xgboost import XGBClassifier, XGBRegressor

# --- SETUP ---
warnings.filterwarnings('ignore')
os.makedirs('../models', exist_ok=True)

# Path Handling
DATA_PATH = '../data/Processed/multi_year_health_data.csv'
if not os.path.exists(DATA_PATH): DATA_PATH = 'data/Processed/multi_year_health_data.csv'

# ==========================================
# PART 1: CHRONIC DISEASE SYSTEM (High-Dimensional)
# ==========================================
print(f"--- Loading Enhanced Dataset from {DATA_PATH} ---")
try:
    df = pd.read_csv(DATA_PATH)
    print(f"Loaded {len(df)} rows with {len(df.columns)} features.")
except:
    print("CRITICAL ERROR: Data file not found. Run data_merging.ipynb first!")
    df = pd.DataFrame()

# 1. DEFINE THE EXPANDED FEATURE SET
X_cols = [
    'Age', 'BMI', 'Sex', 'Income', 'Education', 'MaritalStatus', 'Veteran', 'Employment',
    'Avg_AQI', 'HighBP', 'HighChol', 'GenHealth', 'HealthcareAccess',
    'Smoker', 'HeavyDrinker', 'PhysicalActivity', 'FruitCons', 'VegCons',
    'Blind', 'Deaf', 'CognitiveDiff', 'DiffWalking', 'DiffDressing', 'DiffErrands',
    'PhysicalHealthDays', 'MentalHealthDays'
]

# Robust Feature Selection: Only use columns that actually exist in the CSV
active_cols = [c for c in X_cols if c in df.columns]
print(f"Active Features used for training: {len(active_cols)}")

# 2. DEFINE TARGETS
targets = [
    'HeartAttack', 'Angina', 'Stroke', 'Asthma', 'SkinCancer', 'OtherCancer', 
    'COPD', 'Depression', 'KidneyDisease', 'Diabetes', 'Arthritis'
]

metrics = []

print("\n--- Starting XGBoost Training Pipeline ---")

if not df.empty:
    for target in targets:
        if target not in df.columns: 
            continue
            
        print(f"\n[Target: {target}]")
        
        # Drop rows where Target or Features are missing
        df_clean = df.dropna(subset=active_cols + [target])
        
        # --- SAFETY CHECK 1: Is there data left? ---
        if df_clean.empty:
            print(f"   -> Skipped: No data remaining after cleaning.")
            continue
            
        # --- SAFETY CHECK 2: Are there at least 2 classes? ---
        # The error happened because some diseases might have 0 positive cases left
        unique_classes = df_clean[target].unique()
        if len(unique_classes) < 2:
            print(f"   -> Skipped: Target has only 1 class {unique_classes}. Needs both 0 and 1.")
            continue

        # 3. HANDLE CLASS IMBALANCE
        try:
            rus = RandomUnderSampler(random_state=42)
            X, y = rus.fit_resample(df_clean[active_cols], df_clean[target])
            
            # Split
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
            
            # 4. TRAIN XGBOOST
            model = XGBClassifier(
                n_estimators=100, 
                max_depth=6, 
                learning_rate=0.1, 
                eval_metric='logloss', 
                n_jobs=-1
            )
            model.fit(X_train, y_train)
            
            # 5. EVALUATE
            preds = model.predict(X_test)
            acc = accuracy_score(y_test, preds)
            rec = recall_score(y_test, preds)
            
            print(f"   -> Accuracy: {acc:.1%} | Recall: {rec:.1%}")
            
            # Save
            joblib.dump(model, f'../models/model_{target}.pkl')
            metrics.append({'Disease': target, 'Accuracy': acc, 'Recall': rec})
            
        except Exception as e:
            print(f"   -> Error during training: {e}")

    # Save Metrics
    if metrics:
        pd.DataFrame(metrics).to_csv('../models/training_metrics.csv', index=False)
        print("\n--- Chronic Disease Training Complete ---")
else:
    print("Skipping Chronic Disease Training (No Data).")


# ==========================================
# PART 2: SPECIALIZED MODULES (Parkinson's & Autism)
# ==========================================
print("\n--- Training Specialized Modules ---")

# A. PARKINSON'S
try:
    pk_path = '../data/Processed/parkinsons_clean.csv'
    if not os.path.exists(pk_path): pk_path = 'data/Processed/parkinsons_clean.csv'
    
    if os.path.exists(pk_path):
        df_pk = pd.read_csv(pk_path)
        drop_cols = ['subject#', 'age', 'sex', 'test_time', 'motor_UPDRS', 'total_UPDRS']
        X_pk = df_pk.drop(columns=[c for c in drop_cols if c in df_pk.columns])
        y_pk = df_pk['total_UPDRS']
        
        model_pk = XGBRegressor(n_estimators=100, n_jobs=-1)
        model_pk.fit(X_pk, y_pk)
        joblib.dump(model_pk, '../models/model_parkinsons.pkl')
        print("   -> Parkinson's Model Saved.")
except Exception as e:
    print(f"   -> Skipped Parkinson's: {e}")

# B. AUTISM
try:
    aut_path = '../data/Processed/autism_clean.csv'
    if not os.path.exists(aut_path): aut_path = 'data/Processed/autism_clean.csv'
        
    if os.path.exists(aut_path):
        df_aut = pd.read_csv(aut_path)
        
        # Detect Target
        target = None
        if 'Class' in df_aut.columns: target = 'Class'
        elif 'ASD_Class' in df_aut.columns: target = 'ASD_Class'
        else: target = df_aut.columns[-1]
        
        X_aut = df_aut.drop(columns=[target])
        y_aut = df_aut[target].astype(str).apply(lambda x: 1 if x.lower() in ['yes', 'p', 'positive', '1'] else 0)
        X_aut = pd.get_dummies(X_aut, drop_first=True)
        
        model_aut = XGBClassifier(eval_metric='logloss', n_jobs=-1)
        model_aut.fit(X_aut, y_aut)
        
        joblib.dump(model_aut, '../models/model_autism.pkl')
        joblib.dump(X_aut.columns.tolist(), '../models/autism_features.pkl')
        print(f"   -> Autism Model Saved.")
except Exception as e:
    print(f"   -> Skipped Autism: {e}")

--- Loading Enhanced Dataset from ../data/Processed/multi_year_health_data.csv ---
Loaded 433323 rows with 39 features.
Active Features used for training: 26

--- Starting XGBoost Training Pipeline ---

[Target: HeartAttack]
   -> Accuracy: 76.0% | Recall: 81.7%

[Target: Angina]
   -> Accuracy: 76.2% | Recall: 82.4%

[Target: Stroke]
   -> Accuracy: 73.3% | Recall: 78.8%

[Target: Asthma]
   -> Accuracy: 62.4% | Recall: 59.8%

[Target: SkinCancer]
   -> Skipped: Target has only 1 class [0]. Needs both 0 and 1.

[Target: OtherCancer]
   -> Skipped: Target has only 1 class [0]. Needs both 0 and 1.

[Target: COPD]
   -> Accuracy: 74.6% | Recall: 75.9%

[Target: Depression]
   -> Accuracy: 75.2% | Recall: 74.1%

[Target: KidneyDisease]
   -> Accuracy: 73.6% | Recall: 76.7%

[Target: Diabetes]
   -> Accuracy: 74.8% | Recall: 81.2%

[Target: Arthritis]
   -> Accuracy: 73.6% | Recall: 78.8%

--- Chronic Disease Training Complete ---

--- Training Specialized Modules ---
   -> Parkinson's Mod